# 2.1 理论计算题
给定字符序列 `"ababc"`，采用一阶马尔可夫模型 $p(x_t|x_{t-1})$，使用**拉普拉斯加1平滑**估计条件概率，词汇表 $V=\{\text{a},\text{b},\text{c}\}$，词汇规模 $|V|=3$。

## 一、提取序列转移对
原始序列：$a \to b \to a \to b \to c$
相邻二元转移共4组：$(a,b),(b,a),(a,b),(b,c)$

### 转移计数统计
$count(x_{t-1}\to x_t)$ 表示从前字符 $x_{t-1}$ 转移到后字符 $x_t$ 的次数；
$C(x)$ 表示前字符 $x$ 在转移首位置出现的总次数。

| 前字符 \ 后字符 | a | b | c | 行总计数 $C(x)$ |
|----------------|---|---|---|-----------------|
| a              | 0 | 2 | 0 | 2 |
| b              | 1 | 0 | 1 | 2 |
| c              | 0 | 0 | 0 | 0 |

## 二、拉普拉斯加1平滑公式
$$
p(y|x) = \frac{count(x\to y) + 1}{C(x) + |V|},\quad |V|=3
$$

## 三、概率计算
### 1. $p(\text{a} \mid \text{b})$
$count(b\to a)=1,\ C(b)=2$
$$
p(a|b) = \frac{1+1}{2+3} = \frac{2}{5} = 0.4
$$

### 2. $p(\text{c} \mid \text{b})$
$count(b\to c)=1,\ C(b)=2$
$$
p(c|b) = \frac{1+1}{2+3} = \frac{2}{5} = 0.4
$$

## 四、结果汇总
1. $\boldsymbol{p(\text{a}|\text{b}) = \dfrac{2}{5}}$
2. $\boldsymbol{p(\text{c}|\text{b}) = \dfrac{2}{5}}$

In [1]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 步骤1：转小写，去除标点，仅保留字母和空格
    lower_text = text.lower()
    # 过滤标点符号
    cleaned_chars = []
    for char in lower_text:
        if char.isalpha() or char == " ":
            cleaned_chars.append(char)
    cleaned_text = "".join(cleaned_chars)
    
    # 步骤2：按空格分词，过滤空字符串（多空格情况）
    words = [word for word in cleaned_text.split() if word]
    
    # 步骤3：构建词频统计，按频率降序分配ID（同频按出现先后）
    word_counter = Counter(words)
    # 先按频次倒序，频次相同保留首次出现顺序
    sorted_words = sorted(word_counter.keys(), key=lambda x: (-word_counter[x], words.index(x)))
    word2id = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 步骤4：滑动窗口生成特征与标签
    features = []
    labels = []
    seq_len = len(words)
    # 滑动窗口遍历，窗口起始位置 i 最大为 seq_len - n
    for i in range(seq_len - n):
        window = words[i:i+n]
        next_word = words[i + n]
        features.append(window)
        labels.append(next_word)
    # 最后一组窗口，无后续词，标签为 None
    last_window_start = seq_len - n
    if last_window_start >= 0:
        last_window = words[last_window_start : last_window_start + n]
        features.append(last_window)
        labels.append(None)
    
    return word2id, (features, labels)

# 示例测试
if __name__ == "__main__":
    input_text = "The time machine"
    n = 2
    vocab, (feat, lab) = preprocess_text(input_text, n)
    print("词汇表word2id：", vocab)
    print("特征列表：", feat)
    print("标签列表：", lab)

词汇表word2id： {'the': 0, 'time': 1, 'machine': 2}
特征列表： [['the', 'time'], ['time', 'machine']]
标签列表： ['machine', None]


# 3.1 线性RNN BPTT梯度推导与梯度消失/爆炸分析
## 一、模型前向公式
无偏置线性RNN，无激活函数：
1. 隐状态更新：
$$
h_t = W_{hh} h_{t-1} + W_{hx} x_t
$$
2. 输出层：
$$
o_t = W_{oh} h_t
$$
3. 全局平方损失（长度为$T$的序列）：
$$
L = \frac12 \sum_{t=1}^T \|o_t - y_t\|_2^2
$$

## 二、梯度推导（BPTT沿时间反向传播）
### 1. 输出误差梯度
单步损失 $L_t=\frac12\|o_t-y_t\|^2$，总损失 $L=\sum_{t=1}^T L_t$
$$
\frac{\partial L}{\partial o_t} = o_t - y_t
$$

### 2. 隐状态误差项 $\delta_t = \frac{\partial L}{\partial h_t}$
链式求导：
$$
\delta_t = \frac{\partial o_t}{\partial h_t}^\top \frac{\partial L}{\partial o_t}
= W_{oh}^\top (o_t - y_t)
$$

### 3. 误差沿时间反向递推
$h_t$ 依赖 $h_{t-1}$，反向传递：
$$
\delta_{t-1} = \frac{\partial h_t}{\partial h_{t-1}}^\top \delta_t = W_{hh}^\top \delta_t
$$
递推展开至末尾时刻$T$：
$$
\delta_k = \big(W_{hh}^\top\big)^{T-k} \delta_T
$$

### 4. 损失对 $W_{hh}$ 的梯度
对单步 $t$，矩阵梯度：
$$
\frac{\partial L_t}{\partial W_{hh}} = \delta_t \cdot h_{t-1}^\top
$$
总梯度累加全部时间步：
$$
\boldsymbol{
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t\, h_{t-1}^\top
= \sum_{t=1}^T \Big[ \big(W_{hh}^\top\big)^{T-t} W_{oh}^\top (o_T - y_T) \Big] h_{t-1}^\top
}
$$

## 三、梯度消失与梯度爆炸条件
设转移矩阵 $W_{hh}$ 的**谱半径**（特征值模的最大值）为 $\rho(W_{hh})$：
1. **梯度爆炸**
$\boldsymbol{\rho(W_{hh}) > 1}$
随着回溯时间步增大，$(W_{hh}^\top)^k$ 矩阵元素指数放大，梯度数值剧烈膨胀。

2. **梯度消失**
$\boldsymbol{\rho(W_{hh}) < 1}$
随着回溯时间步增大，$(W_{hh}^\top)^k$ 矩阵元素指数趋近于0，远距离时序的梯度贡献丢失，无法学习长依赖。

3. 临界稳定：$\rho(W_{hh})=1$，梯度幅度不随序列长度指数缩放。

### 核心本质
线性RNN时序依赖传递等价于重复乘以 $W_{hh}$，矩阵幂的缩放由谱半径控制，是梯度不稳定的根源。

In [2]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单步前向传播
    参数：
        x_t: (batch_size, input_size)
        h_prev: (batch_size, hidden_size)
        W_hx: (input_size, hidden_size)
        W_hh: (hidden_size, hidden_size)
        b_h: (1, hidden_size)
    返回：
        h_t: 当前隐状态 (batch, hidden)
        cache: 缓存前向中间变量，反向传播使用
    """
    z_t = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h
    h_t = np.tanh(z_t)
    cache = (x_t, h_prev, W_hx, W_hh, b_h, z_t, h_t)
    return h_t, cache


def rnn_backward(dh_next, cache):
    """
    RNN单步反向传播，计算全部梯度
    参数：
        dh_next: 上游梯度 dL/dh_t (batch, hidden)
        cache: 前向传播保存的中间变量
    返回：
        dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    x_t, h_prev, W_hx, W_hh, b_h, z_t, h_t = cache
    batch_size = x_t.shape[0]

    # tanh导数
    dz_t = dh_next * (1 - np.square(h_t))

    # 输入梯度 dx_t
    dx_t = np.dot(dz_t, W_hx.T)
    # 上一时刻隐状态梯度 dh_prev
    dh_prev = np.dot(dz_t, W_hh.T)
    # 权重梯度
    dW_hx = np.dot(x_t.T, dz_t)
    dW_hh = np.dot(h_prev.T, dz_t)
    # 偏置梯度：沿batch求和
    db_h = np.sum(dz_t, axis=0, keepdims=True)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 测试示例
if __name__ == "__main__":
    # 超参数
    B = 2    # batch_size
    I = 3    # input_size
    H = 4    # hidden_size

    # 随机初始化输入与权重
    x_t = np.random.randn(B, I)
    h_prev = np.random.randn(B, H)
    W_hx = np.random.randn(I, H)
    W_hh = np.random.randn(H, H)
    b_h = np.random.randn(1, H)

    # 前向传播
    h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("前向输出 h_t shape:", h_t.shape)

    # 模拟上游梯度 dL/dh_t
    dh_next = np.random.randn(B, H)
    # 反向传播求梯度
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache)

    print("dx_t shape:", dx_t.shape)
    print("dh_prev shape:", dh_prev.shape)
    print("dW_hx shape:", dW_hx.shape)
    print("dW_hh shape:", dW_hh.shape)
    print("db_h shape:", db_h.shape)

前向输出 h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (3, 4)
dW_hh shape: (4, 4)
db_h shape: (1, 4)


# 4.1 深度双向RNN参数总数推导
## 1. 模型设定
- $L$：RNN层数；$H$：单层单向隐单元数；$D$：输入维度；$O$：输出维度
- 双向RNN每层包含前向、后向两套独立RNN权重
- 第1层输入维度为$D$；第2~L层输入维度为$2H$（上层双向隐状态拼接）
- 输出层输入维度为$2H$，包含权重与偏置

## 2. 单层单向RNN参数公式
输入维度为$I$时，单向RNN单元参数：
$$
P_{\text{single}}(I) = I\cdot H + H\cdot H + H
$$
分别对应输入权重$W_{hx}$、循环权重$W_{hh}$、隐层偏置$b_h$。

## 3. 分层参数计算
### (1) 第1层双向RNN（输入$I=D$）
$$
P_1 = 2\cdot\left(DH + H^2 + H\right)
$$
### (2) 第$l\ge2$层双向RNN（输入$I=2H$）
$$
P_l = 2\cdot\left(2H\cdot H + H^2 + H\right) = 6H^2 + 2H
$$
### (3) $L$层RNN总参数
$$
\begin{align*}
P_{\text{rnn}} &= 2(DH+H^2+H) + (L-1)(6H^2+2H) \\
&= 2DH + (6L-4)H^2 + 2LH
\end{align*}
$$

### (4) 输出层参数
$$
P_{\text{out}} = 2H\cdot O + O
$$

## 4. 全局总参数表达式
$$
\boldsymbol{
P_{\text{total}} = 2DH + (6L-4)H^2 + 2LH + 2HO + O
}
$$

### 各项含义
1. $2DH$：第一层双向输入投影权重；
2. $(6L-4)H^2$：全部RNN层循环权重与深层输入投影权重；
3. $2LH$：所有RNN单元的偏置项；
4. $2HO$：输出层全连接权重；
5. $O$：输出层偏置项。

In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        # 双向单层RNN，batch_first=False
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=False
        )

    def forward(self, X):
        """
        :param X: 输入序列 (seq_len, batch, input_dim)
        :return:
            outputs: 每一步拼接隐状态 (seq_len, batch, 2*hidden_dim)
            final_hidden: 全局序列表示 (batch, 2*hidden_dim)
        """
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (num_layers * num_directions, batch, hidden_dim)
        outputs, h_n = self.rnn(X)

        # 分离前向、后向最终隐状态
        h_forward = h_n[0]   # (batch, hidden_dim) 前向最后一步
        h_backward = h_n[1]  # (batch, hidden_dim) 后向最后一步
        # 拼接得到全局序列表征
        final_hidden = torch.cat([h_forward, h_backward], dim=-1)  # (batch, 2*hidden_dim)

        return outputs, final_hidden


# 测试示例
if __name__ == "__main__":
    # 超参数
    seq_len = 10
    batch_size = 4
    input_dim = 5
    hidden_dim = 8

    # 构造输入 (seq_len, batch, input_dim)
    x = torch.randn(seq_len, batch_size, input_dim)
    # 初始化编码器
    encoder = BiRNNEncoder(input_dim, hidden_dim)
    # 前向传播
    seq_output, global_h = encoder(x)

    print("时序输出 outputs shape:", seq_output.shape)
    print("全局表征 final_hidden shape:", global_h.shape)

时序输出 outputs shape: torch.Size([10, 4, 16])
全局表征 final_hidden shape: torch.Size([4, 16])


# 5.1 Skip-gram 负采样对数似然损失推导
## 1. 基础概率定义
中心词向量 $\boldsymbol{v}_c$，上下文输出向量 $\boldsymbol{u}_w$，sigmoid条件概率：
$$
P(w \mid w_c) = \sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_w) = \frac{1}{1+\exp\left(-\boldsymbol{v}_c^\top \boldsymbol{u}_w\right)}
$$
- 正样本：目标上下文 $w_o$，向量 $\boldsymbol{u}_o$
- 负样本：$K$ 个噪声词 $w_{n_1},\dots,w_{n_K}$，对应向量 $\boldsymbol{u}_{n_k}$

## 2. 联合似然
目标：最大化正样本概率，同时最小化所有负样本概率：
$$
L = P(w_o \mid w_c) \cdot \prod_{k=1}^K \Big[1-P(w_{n_k} \mid w_c)\Big]
$$

## 3. 对数似然目标函数（最大化）
对似然取自然对数，得到模型优化目标：
$$
\boldsymbol{J} = \log\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o) + \sum_{k=1}^K \log\left(1-\sigma\left(\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k}\right)\right)
$$
若定义**损失函数（最小化）**，添加负号：
$$
\mathcal{L} = -\log\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o) - \sum_{k=1}^K \log\left(1-\sigma\left(\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k}\right)\right)
$$

## 4. 噪声分布与负样本采样方法
### (1) 噪声分布 $P_n(w)$
Word2Vec标准分布，基于词频的3/4次幂平滑：
$$
P_n(w) = \frac{\text{count}(w)^{\frac{3}{4}}}{\sum_{w' \in V}\text{count}(w')^{\frac{3}{4}}}
$$
$V$：全部词汇表；$\text{count}(w)$：词 $w$ 在语料中的出现次数。

### (2) 采样步骤
1. 预处理语料，统计每个单词频次，计算噪声分布 $P_n(w)$；
2. 对每一组 $(w_c,w_o)$，循环采样 $K$ 次：
   - 根据 $P_n(w)$ 随机抽取候选负词；
   - 若抽取到 $w_o$（与正样本重复），丢弃该样本并重新采样；
3. 最终得到 $K$ 个不与正样本重复的负样本，代入损失计算。

### (3) 采样设计作用
1. 大幅降低计算开销，替代原始Softmax全局词汇求和；
2. 高频词更易作为负样本，平衡训练梯度，提升词向量质量。

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(batch_ctx, targets, W, W_out):
    """
    CBOW前向传播+完整softmax交叉熵损失计算
    参数：
        batch_ctx: 上下文索引批次 (batch_size, context_size)
        targets: 中心词目标索引 (batch_size,)
        W: 输入嵌入矩阵 (V, d)
        W_out: 输出权重矩阵 (d, V)
    返回：
        loss: 批量平均交叉熵标量损失
    """
    batch_size, context_size = batch_ctx.shape
    
    # 1. 上下文词嵌入 [batch, context_size, d]
    ctx_emb = W[batch_ctx]
    
    # 2. 求平均得到隐藏层 h [batch, d]
    h = torch.sum(ctx_emb, dim=1) / context_size
    
    # 3. 计算输出得分 logits [batch, V]
    logits = torch.matmul(h, W_out)
    
    # 4. 交叉熵损失（内部包含softmax）
    loss = F.cross_entropy(logits, targets)
    
    return loss


# 测试示例
if __name__ == "__main__":
    V = 100    # 词汇表大小
    d = 16     # 嵌入维度
    batch_size = 4
    context_size = 2
    
    # 随机初始化权重
    W = torch.randn(V, d)
    W_out = torch.randn(d, V)
    
    # 构造输入：每个样本2个上下文词索引
    batch_ctx = torch.randint(0, V, (batch_size, context_size))
    # 中心词目标索引
    targets = torch.randint(0, V, (batch_size,))
    
    loss_val = cbow_forward_loss(batch_ctx, targets, W, W_out)
    print("CBOW 批量平均损失值：", loss_val.item())

CBOW 批量平均损失值： 5.9513139724731445


# 6.1 缩放点积注意力分步计算
## 已知参数
$Q\in\mathbb{R}^{2\times4},\ K\in\mathbb{R}^{3\times4},\ V\in\mathbb{R}^{3\times5},\ d_k=4,\ \sqrt{d_k}=2$

缩放点积注意力标准公式：
$$
\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

## 步骤1：计算缩放得分矩阵
1. 计算Query与Key原始相似度矩阵：
$$
S_{\text{raw}} = QK^\top
$$
维度：$S_{\text{raw}} \in \mathbb{R}^{2\times3}$

2. 除以 $\sqrt{d_k}=2$ 缩放，得到得分矩阵：
$$
\text{score} = \frac{QK^\top}{2}
$$
维度：$\text{score} \in \mathbb{R}^{2\times3}$

## 步骤2：逐行Softmax得到注意力权重
对得分矩阵**每一行**独立执行Softmax归一化，权重矩阵元素：
$$
w_{i,j} = \frac{\exp(\text{score}_{i,j})}{\sum_{m=1}^3 \exp(\text{score}_{i,m})}
$$
记注意力权重矩阵为 $W_{\text{attn}}$，维度 $W_{\text{attn}} \in \mathbb{R}^{2\times3}$，矩阵每行所有元素之和为1。

## 步骤3：加权求和得到注意力输出
注意力权重矩阵与值矩阵 $V$ 做矩阵乘法：
$$
O = W_{\text{attn}} \cdot V
$$
维度匹配：$(2 \times 3) \times (3 \times 5) = 2 \times 5$，最终输出矩阵 $O \in \mathbb{R}^{2\times5}$。

## 维度流转汇总
1. $QK^\top$：$2 \times 3$
2. 缩放得分 $\text{score}$：$2 \times 3$
3. 注意力权重 $W_{\text{attn}}$：$2 \times 3$
4. 注意力输出 $O$：$2 \times 5$

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_heads = 2
        self.d_model = 4
        self.d_k = self.d_model // self.num_heads  # d_k=2
        
        # Q/K/V 投影层，统一映射到 d_model 维度，后续分头
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        # 输出融合线性层
        self.w_o = nn.Linear(self.d_model, self.d_model)

    def scaled_dot_product_attention(self, q, k, v):
        """
        缩放点积注意力，输入形状: (seq_len, batch, num_heads, d_k)
        """
        dk = q.size(-1)
        # 计算得分 score = QK^T / sqrt(d_k)
        score = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(dk, dtype=torch.float32))
        attn_weight = F.softmax(score, dim=-1)
        out = torch.matmul(attn_weight, v)
        return out

    def forward(self, X):
        """
        :param X: 输入序列 (seq_len, batch, d_model)
        :return: 多头注意力输出 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 1. 线性投影得到 Q, K, V
        Q = self.w_q(X)
        K = self.w_k(X)
        V = self.w_v(X)
        
        # 2. 分头：拆分为 num_heads 个头，维度变换 (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_k)
        
        # 3. 单头缩放点积注意力
        head_out = self.scaled_dot_product_attention(Q, K, V)
        
        # 4. 拼接所有头：合并头维度，还原 d_model
        concat = head_out.contiguous().view(seq_len, batch, self.d_model)
        
        # 5. 输出线性层，最终输出
        output = self.w_o(concat)
        return output


# 测试示例
if __name__ == "__main__":
    seq_len = 5
    batch = 3
    d_model = 4
    # 构造输入 (seq_len, batch, d_model)
    x = torch.randn(seq_len, batch, d_model)
    mha = MultiHeadAttention()
    res = mha(x)
    print("输入形状:", x.shape)
    print("输出形状:", res.shape)

输入形状: torch.Size([5, 3, 4])
输出形状: torch.Size([5, 3, 4])
